In [0]:
#Reuse or recreate a small DataFrame with a mix of types and some nulls
from pyspark.sql.functions import col, count, when, countDistinct, min, max

data = [(1, "Alice", "IT", 55000),
        (2, "Ben", "HR", None),
        (3, "Cara", "IT", 62000),
        (4, "Dev", None, 45000),
        (5, "Alice", "IT", 55000)]
columns = ["id", "name", "department", "salary"]

employees = spark.createDataFrame(data, columns)
employees.display()

In [0]:
# Basic profilling - one column at a time
employees.select(count(col("salary")).alias("non_null_count"),
                 countDistinct(col("salary")).alias("distinct_count"),
                 min(col("salary")).alias("min"),
                 max(col("salary")).alias("max")
                 ).display()


In [0]:
#The one-pass null check across every column
null_counts = employees.select([
    count(when(col(c).isNull(), c)).alias(c) for c in employees.columns
])
null_counts.display()

#Output Check: salary and department should each show 1 null.

In [0]:
#Min/max, only for numeric columns
from pyspark.sql.types import NumericType

for field in employees.schema.fields:
    if isinstance(field.dataType, NumericType):
        employees.select(min(col(field.name)).alias("min"), max(col(field.name)).alias("max")).show()
    else:
        print(f"{field.name}: not numeric, skipping min/max")

In [0]:
# Build the reusable profile_column() function
def profile_column(df, column_name):
    total = df.count()
    non_null = df.filter(col(column_name).isNotNull()).count()
    distinct = df.select(column_name).distinct().count()
    return {
        "column":column_name,
        "total_rows":total,
        "non_null":non_null,
        "distinct":distinct
    }
for c in employees.columns:
    print(profile_column(employees, c))


In [0]:
# Try Built In comparison
employees.summary().show()